# MPS viability check --- read-only, no state changes

Checks whether this Databricks cluster can plausibly run
`nvidia-smi -i 0 -c EXCLUSIVE_PROCESS` + `nvidia-cuda-mps-control -d`
at all, before attempting either. Nothing here changes GPU compute
mode or starts any daemon. Run this any time, including while another
job holds the GPU -- these commands do not need a CUDA context.

**The actual mode switch is a separate, later step** (a follow-up notebook), run only once nothing else is using the GPU -- flipping compute mode under an active CUDA context is exactly the kind of thing that fails messily or disrupts the running job.

In [ ]:
import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    print(f'$ {cmd}')
    print(f'  exit={r.returncode}')
    if r.stdout.strip():
        print('  stdout:')
        for line in r.stdout.strip().splitlines():
            print('   ', line)
    if r.stderr.strip():
        print('  stderr:')
        for line in r.stderr.strip().splitlines():
            print('   ', line)
    print()
    return r

## 1. Who are we, and do we have privileged access

In [ ]:
sh('whoami')
sh('id')
sh('sudo -n true && echo SUDO_OK || echo NO_PASSWORDLESS_SUDO')

## 2. Driver, compute mode, and whether the MPS binaries exist

In [ ]:
sh('nvidia-smi -q | head -40')
sh('nvidia-smi -i 0 -q -d COMPUTE | grep -A2 "Compute Mode"')
sh('which nvidia-cuda-mps-control || echo NOT_FOUND')
sh('which nvidia-cuda-mps-server || echo NOT_FOUND')

## 3. Is anything already holding a CUDA context on this GPU

In [ ]:
sh('nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv')
sh('nvidia-smi -i 0 -q -d MEMORY | grep -A4 "FB Memory Usage"')

## Verdict

Read the four sections above and decide:
- `NO_PASSWORDLESS_SUDO` above means the compute-mode switch will need `sudo -S` with a password this notebook does not have, or root is genuinely unavailable here.
- `NOT_FOUND` on either MPS binary means the runtime image does not ship the MPS control/server tools at all -- installing them is a different, larger task than flipping a mode.
- A non-empty compute-apps list means something (very likely this very kernel, or another job) already holds a context -- the mode switch must wait until that list is empty.